# Bird Feeder AI: EfficientNet-B2 ONNX to Hailo HEF Conversion

Converts the fine-tuned EfficientNet-B2 bird species classifier (555 NABirds classes, 260x260 input) from ONNX to Hailo HEF format for deployment on the Hailo-10H NPU (AI HAT+ 2).

Uses the Hailo DFC `ClientRunner` API because `efficientnet_b2` is not in the Hailo Model Zoo. This script handles:
- ONNX parsing with correct input shape (1, 3, 260, 260)
- On-device ImageNet normalization (baked into the HEF so the NPU normalizes on-chip)
- INT8 quantization with calibration images (raw 0-255 pixels, NHWC format)
- Compilation for Hailo-10H

**Before running, upload 3 files to a Google Drive folder called `hailo_convert/`:**
1. `hailo_dataflow_compiler-5.3.0-py3-none-linux_x86_64.whl` (from [Hailo Developer Zone](https://hailo.ai/developer-zone/software-downloads/))
2. `efficientnet_b2_birds.onnx` (from `models/onnx/`)
3. `calib_nabirds.zip` (pre-built calibration zip, from project root)

Your Drive folder should look like:
```
My Drive/
  hailo_convert/
    hailo_dataflow_compiler-5.3.0-py3-none-linux_x86_64.whl
    efficientnet_b2_birds.onnx
    calib_nabirds.zip
```

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import glob
import os

drive.mount('/content/drive')

# Verify files exist
DRIVE_DIR = '/content/drive/MyDrive/hailo_convert'
print('Files in hailo_convert/:')
!ls -lh {DRIVE_DIR}/

## Step 2: Install System Dependencies

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3-dev python3-distutils python3-tk libfuse2 graphviz libgraphviz-dev

## Step 3: Install the Hailo DFC

In [ ]:
# Find the wheel file
whl_files = glob.glob(f'{DRIVE_DIR}/hailo_dataflow_compiler*.whl')
if not whl_files:
    raise FileNotFoundError(f'No DFC wheel found in {DRIVE_DIR}/')
whl_file = whl_files[0]
print(f'Using: {whl_file}')

# Install DFC in a virtualenv (required due to a known DFC packaging bug)
!pip install -q virtualenv
!virtualenv hailo_env
!hailo_env/bin/pip install -q {whl_file}

# Pin compatible numpy/scipy/Pillow versions
!hailo_env/bin/pip install -q "numpy>=1.23,<2.0" "scipy>=1.10,<1.12" "Pillow>=9.0"

# Verify installation
!hailo_env/bin/python -c "from hailo_sdk_client import ClientRunner; print('DFC installed successfully')"

## Step 4: Set Up Files

In [ ]:
os.makedirs('/content/onnx', exist_ok=True)
os.makedirs('/content/hef', exist_ok=True)
os.makedirs('/content/calib_images', exist_ok=True)

# Copy ONNX model from Drive (faster than reading directly from Drive during compilation)
!cp {DRIVE_DIR}/efficientnet_b2_birds.onnx /content/onnx/
print('ONNX model copied')
!ls -lh /content/onnx/

# Extract calibration images
!unzip -q -o {DRIVE_DIR}/calib_nabirds.zip -d /content/calib_images
num = len(glob.glob('/content/calib_images/*.jpg') + glob.glob('/content/calib_images/*.png'))
print(f'Extracted {num} calibration images')

## Step 5: Compile EfficientNet-B2 to HEF

Runs the full pipeline: Parse ONNX → Add on-device normalization → INT8 quantization → Compile HEF.

The normalization is baked into the HEF so the Hailo NPU handles `(pixel - mean) / std` on-chip. At inference time, the host just sends raw uint8 pixels.

In [ ]:
%%writefile /content/convert_hailo.py
"""ONNX to HEF conversion for EfficientNet-B2 bird classifier."""

import logging
import random
from pathlib import Path

import numpy as np
from PIL import Image
from hailo_sdk_client import ClientRunner

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# ImageNet normalization scaled to 0-255 range for Hailo on-device normalization
HAILO_MEAN = [123.675, 116.28, 103.53]
HAILO_STD = [58.395, 57.12, 57.375]

INPUT_SIZE = 260
RESIZE_SIZE = 292  # INPUT_SIZE + 32, matches PyTorch val transforms


def load_calibration_images(calib_dir, num_images=1024):
    """Load RAW (unnormalized) calibration images in NHWC format, values 0-255."""
    image_paths = sorted(
        list(Path(calib_dir).rglob('*.jpg'))
        + list(Path(calib_dir).rglob('*.jpeg'))
        + list(Path(calib_dir).rglob('*.png'))
    )
    if not image_paths:
        raise FileNotFoundError(f'No images found in {calib_dir}')

    random.seed(42)
    if len(image_paths) > num_images:
        image_paths = random.sample(image_paths, num_images)
    logger.info(f'Loading {len(image_paths)} calibration images')

    images = []
    for path in image_paths:
        try:
            img = Image.open(path).convert('RGB')
            w, h = img.size
            if w < h:
                new_w, new_h = RESIZE_SIZE, int(RESIZE_SIZE * h / w)
            else:
                new_w, new_h = int(RESIZE_SIZE * w / h), RESIZE_SIZE
            img = img.resize((new_w, new_h), Image.BILINEAR)
            left = (new_w - INPUT_SIZE) // 2
            top = (new_h - INPUT_SIZE) // 2
            img = img.crop((left, top, left + INPUT_SIZE, top + INPUT_SIZE))
            arr = np.array(img, dtype=np.float32)  # (260, 260, 3), values 0-255
            images.append(arr)
        except Exception as e:
            logger.warning(f'Skipping {path}: {e}')
    
    logger.info(f'Loaded {len(images)} images ({INPUT_SIZE}x{INPUT_SIZE}, NHWC, 0-255)')
    return np.array(images, dtype=np.float32)


# --- Stage 1: Parse ONNX ---
logger.info('Stage 1: Parsing ONNX model')
runner = ClientRunner(hw_arch='hailo10h')
runner.translate_onnx_model(
    '/content/onnx/efficientnet_b2_birds.onnx',
    'efficientnet_b2_birds',
    start_node_names=['input'],
    end_node_names=['output'],
    net_input_shapes={'input': [1, 3, 260, 260]},
)
runner.save_har('/content/hef/efficientnet_b2_birds_parsed.har')
logger.info('  Parsed HAR saved')

# --- Stage 2: Load model script (on-device normalization + avgpool fix) ---
# The avgpool2 layer has 130x130 spatial dims (from a Squeeze-and-Excitation block).
# The Hailo NPU can't quantize avgpools with shift delta > 2, so we split the
# large avgpool into two smaller ones: 130x130 -> 26x26 -> 1x1.
# Division factors must evenly divide the spatial dims (130 / 5 = 26).
logger.info('Stage 2: Adding on-device ImageNet normalization + avgpool reduction')
model_script = (
    f'normalization1 = normalization({HAILO_MEAN}, {HAILO_STD})\n'
    'pre_quantization_optimization(global_avgpool_reduction, layers=efficientnet_b2_birds/avgpool2, division_factors=[5, 5])\n'
)
runner.load_model_script(model_script)

# --- Stage 3: Load calibration data and quantize ---
logger.info('Stage 3: Loading calibration images')
calib_data = load_calibration_images('/content/calib_images', num_images=1024)

logger.info('Stage 4: Quantizing to INT8 (this may take several minutes)...')
runner.optimize(calib_data)
runner.save_har('/content/hef/efficientnet_b2_birds_quantized.har')
logger.info('  Quantized HAR saved')

# --- Stage 5: Compile to HEF ---
logger.info('Stage 5: Compiling to HEF for hailo10h...')
hef_data = runner.compile()
hef_path = '/content/hef/efficientnet_b2_birds.hef'
with open(hef_path, 'wb') as f:
    f.write(hef_data)

import os
size_mb = os.path.getsize(hef_path) / (1024 * 1024)
logger.info(f'Done! HEF saved: {hef_path} ({size_mb:.1f} MB)')

In [ ]:
%%time
!hailo_env/bin/python /content/convert_hailo.py

## Step 6: Copy HEF Back to Google Drive

In [ ]:
print('Compiled HEF:')
!ls -lh /content/hef/*.hef

# Copy to Google Drive for persistent storage
!cp /content/hef/efficientnet_b2_birds.hef {DRIVE_DIR}/
print(f'\nCopied to {DRIVE_DIR}/efficientnet_b2_birds.hef')
print('Download it from Google Drive to your project: models/hef/')

## Done!

The HEF file is saved to your Google Drive at `hailo_convert/efficientnet_b2_birds.hef`.

Download it and place in your project:

```
models/hef/
  yolov11n.hef                <- pre-compiled from Hailo Model Zoo (bird detection)
  efficientnet_b2_birds.hef   <- just compiled (bird species classifier, 555 classes)
```

Run on your Raspberry Pi:

```bash
python -m src.pipeline.pipeline --mode hailo
```

**Note:** The HEF has ImageNet normalization baked in. The Hailo inference backend
should send raw uint8 pixels (0-255) — do NOT apply PyTorch-style normalization
on the host side.